# 03 — Canonical ML data smoke

This notebook performs the first real-data checks for the
rebuilt TorNet ML pipeline.

It:

1. loads the ten canonical frame manifests;
2. validates the expected 812,432-frame index;
3. reports class balance without modifying the index;
4. selects one deterministic positive training frame;
5. extracts only its NetCDF member from the 2013 archive;
6. constructs the DBZ/VEL tensor using named dimensions;
7. verifies that the NetCDF label matches the manifest label.

It does not create a validation split, calculate
normalization statistics, or train a model.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")


In [ ]:
from pathlib import Path

PACKAGE_VERSION = "0.1.4"
BACKUP_ROOT = Path(
    "/content/drive/MyDrive/TorNet_Backup"
)
PACKAGE_PATH = (
    BACKUP_ROOT
    / "packages"
    / (
        "tornet_detection-"
        f"{PACKAGE_VERSION}-py3-none-any.whl"
    )
)
MANIFESTS_ROOT = BACKUP_ROOT / "manifests"
ARCHIVE_PATH = BACKUP_ROOT / "tornet_2013.tar.gz"

required_paths = [
    PACKAGE_PATH,
    MANIFESTS_ROOT,
    ARCHIVE_PATH,
]

missing = [
    str(path)
    for path in required_paths
    if not path.exists()
]

if missing:
    raise FileNotFoundError(
        "Required Drive paths are missing: "
        + ", ".join(missing)
    )

print("package:", PACKAGE_PATH)
print("manifests:", MANIFESTS_ROOT)
print("archive:", ARCHIVE_PATH)


In [ ]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-deps",
        "--force-reinstall",
        str(PACKAGE_PATH),
    ],
    check=True,
)


In [ ]:
import tornado_detection

if tornado_detection.__version__ != PACKAGE_VERSION:
    raise RuntimeError(
        "Unexpected installed package version: "
        f"{tornado_detection.__version__} != "
        f"{PACKAGE_VERSION}. Restart the runtime and "
        "rerun the notebook from the beginning."
    )

print(
    "tornado_detection version:",
    tornado_detection.__version__,
)


In [ ]:
from tornado_detection.data import (
    EXPECTED_CANONICAL_FRAME_COUNT,
    load_canonical_frame_index,
    summarize_canonical_frame_index,
)

frame_index = load_canonical_frame_index(
    MANIFESTS_ROOT
)

assert (
    len(frame_index)
    == EXPECTED_CANONICAL_FRAME_COUNT
)
assert frame_index["frame_id"].is_unique

print(
    "canonical frames:",
    f"{len(frame_index):,}",
)
print(
    "canonical files:",
    f"{frame_index['file_id'].nunique():,}",
)


In [ ]:
summaries = summarize_canonical_frame_index(
    frame_index
)

for name, summary in summaries.items():
    print()
    print(f"=== {name} ===")
    print(summary.to_string(index=False))


In [ ]:
source_selection = (
    frame_index[
        [
            "year",
            "manifest_source",
            "manifest_directory",
        ]
    ]
    .drop_duplicates()
    .sort_values("year")
    .reset_index(drop=True)
)

print(
    source_selection.to_string(index=False)
)


In [ ]:
sample_candidates = (
    frame_index.loc[
        frame_index["year"].eq(2013)
        & frame_index["split"].eq("train")
        & frame_index["frame_label"].eq(1)
    ]
    .sort_values(
        [
            "archive_member",
            "frame_index",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)

if sample_candidates.empty:
    raise RuntimeError(
        "No positive 2013 training frame was found"
    )

sample = sample_candidates.iloc[0]

sample_fields = [
    "frame_id",
    "file_id",
    "archive_member",
    "split",
    "year",
    "category",
    "event_group_id",
    "frame_index",
    "frame_time_utc",
    "frame_label",
    "dbz_finite_fraction",
    "vel_finite_fraction",
    "range_folded_fraction",
]

print(sample[sample_fields].to_string())


In [ ]:
import shutil
import tarfile

LOCAL_ARCHIVE_PATH = Path(
    "/content/tornet_2013.tar.gz"
)
LOCAL_SAMPLE_DIRECTORY = Path(
    "/content/tornet_frame_smoke"
)
LOCAL_SAMPLE_PATH = (
    LOCAL_SAMPLE_DIRECTORY / "sample.nc"
)

usage = shutil.disk_usage("/content")
required_bytes = ARCHIVE_PATH.stat().st_size

print(
    "available local GiB:",
    round(usage.free / (1024 ** 3), 2),
)
print(
    "archive GiB:",
    round(required_bytes / (1024 ** 3), 2),
)

if usage.free < required_bytes * 1.25:
    raise RuntimeError(
        "Insufficient Colab-local space to stage "
        "the 2013 archive safely"
    )

if LOCAL_ARCHIVE_PATH.exists():
    LOCAL_ARCHIVE_PATH.unlink()

if LOCAL_SAMPLE_DIRECTORY.exists():
    shutil.rmtree(
        LOCAL_SAMPLE_DIRECTORY
    )

LOCAL_SAMPLE_DIRECTORY.mkdir(
    parents=True,
    exist_ok=False,
)

shutil.copyfile(
    ARCHIVE_PATH,
    LOCAL_ARCHIVE_PATH,
)

if (
    LOCAL_ARCHIVE_PATH.stat().st_size
    != ARCHIVE_PATH.stat().st_size
):
    raise RuntimeError(
        "Local archive size differs from Drive"
    )

print(
    "staged archive bytes:",
    LOCAL_ARCHIVE_PATH.stat().st_size,
)


In [ ]:
archive_member = str(
    sample["archive_member"]
)

with tarfile.open(
    LOCAL_ARCHIVE_PATH,
    mode="r:gz",
) as archive:
    try:
        member = archive.getmember(
            archive_member
        )
    except KeyError as exc:
        raise FileNotFoundError(
            "Manifest-selected member is absent "
            f"from archive: {archive_member}"
        ) from exc

    if not member.isfile():
        raise RuntimeError(
            "Manifest-selected archive member "
            "is not a regular file"
        )

    source_file = archive.extractfile(member)

    if source_file is None:
        raise RuntimeError(
            "Could not open manifest-selected "
            "archive member"
        )

    with (
        source_file,
        LOCAL_SAMPLE_PATH.open("wb")
        as destination,
    ):
        shutil.copyfileobj(
            source_file,
            destination,
            length=1024 * 1024,
        )

if (
    LOCAL_SAMPLE_PATH.stat().st_size
    != member.size
):
    raise RuntimeError(
        "Extracted NetCDF size differs from tar "
        f"metadata: "
        f"{LOCAL_SAMPLE_PATH.stat().st_size} "
        f"!= {member.size}"
    )

print("member:", archive_member)
print(
    "NetCDF bytes:",
    LOCAL_SAMPLE_PATH.stat().st_size,
)


In [ ]:
import numpy as np
import xarray as xr

frame_index_in_file = int(
    sample["frame_index"]
)
manifest_label = int(
    sample["frame_label"]
)

with xr.open_dataset(
    LOCAL_SAMPLE_PATH,
    engine="netcdf4",
) as dataset:
    print("dataset dimensions:")
    print(dict(dataset.sizes))

    for variable_name in (
        "DBZ",
        "VEL",
    ):
        actual_dimensions = tuple(
            dataset[
                variable_name
            ].dims
        )
        expected_dimensions = (
            "time",
            "azimuth",
            "range",
            "sweep",
        )

        if (
            actual_dimensions
            != expected_dimensions
        ):
            raise AssertionError(
                f"{variable_name} dimensions "
                f"{actual_dimensions} != "
                f"{expected_dimensions}"
            )

    dbz = (
        dataset["DBZ"]
        .isel(
            time=frame_index_in_file
        )
        .transpose(
            "azimuth",
            "range",
            "sweep",
        )
        .values
        .astype(
            np.float32,
            copy=False,
        )
    )

    vel = (
        dataset["VEL"]
        .isel(
            time=frame_index_in_file
        )
        .transpose(
            "azimuth",
            "range",
            "sweep",
        )
        .values
        .astype(
            np.float32,
            copy=False,
        )
    )

    netcdf_label = int(
        dataset["frame_labels"]
        .isel(
            time=frame_index_in_file
        )
        .item()
    )

tensor = np.concatenate(
    [
        dbz,
        vel,
    ],
    axis=-1,
)

assert dbz.shape == (120, 240, 2)
assert vel.shape == (120, 240, 2)
assert tensor.shape == (120, 240, 4)
assert tensor.dtype == np.float32
assert netcdf_label == manifest_label

print("DBZ shape:", dbz.shape)
print("VEL shape:", vel.shape)
print("tensor shape:", tensor.shape)
print("tensor dtype:", tensor.dtype)
print("manifest label:", manifest_label)
print("NetCDF label:", netcdf_label)
print(
    "finite fraction by channel:",
    [
        float(
            np.isfinite(
                tensor[:, :, channel]
            ).mean()
        )
        for channel in range(
            tensor.shape[-1]
        )
    ],
)


In [ ]:
import matplotlib.pyplot as plt

channel_names = [
    "DBZ sweep 0",
    "DBZ sweep 1",
    "VEL sweep 0",
    "VEL sweep 1",
]

figure, axes = plt.subplots(
    2,
    2,
    figsize=(14, 8),
    constrained_layout=True,
)

for channel, axis in enumerate(
    axes.ravel()
):
    image = axis.imshow(
        tensor[:, :, channel],
        origin="lower",
        aspect="auto",
    )
    axis.set_title(
        channel_names[channel]
    )
    axis.set_xlabel("range")
    axis.set_ylabel("azimuth")
    figure.colorbar(
        image,
        ax=axis,
        shrink=0.8,
    )

figure.suptitle(
    f"{sample['frame_id']} | "
    f"frame_label={manifest_label}"
)
plt.show()


In [ ]:
LOCAL_SAMPLE_PATH.unlink(
    missing_ok=True
)
LOCAL_ARCHIVE_PATH.unlink(
    missing_ok=True
)
LOCAL_SAMPLE_DIRECTORY.rmdir()

print("Removed Colab-local smoke artifacts")
